# 13 · 🏁 O Grande Benchmark (Capstone)

**Pré-requisito**: variável — cada célula requer um perfil diferente.

---

🎯 **Objetivo final do laboratório:** executar a **mesmíssima** agregação Gold
(`vendas` join-broadcast `empresas`, agrupado por `setor`/`ano`/`mes`) em todas
as 4 arquiteturas e comparar os tempos.

📌 **As 4 arquiteturas (Casos A-D):**
| Nível | Cluster | Armazenamento | Conexão | Docker |
|---|---|---|---|---|
| **A** — `local[*]` | Nenhum (single process) | Disco local | Direta | Não precisa |
| **B** — Standalone+Connect | Spark Standalone | Volume Docker compartilhado | Spark Connect (gRPC) | `make up-cluster` |
| **C** — YARN+HDFS | Hadoop YARN | HDFS (webhdfs://) | Client mode | `make up-hadoop` |
| **D** — Standalone+S3 | Spark Standalone | RustFS (s3a://) | Spark Connect (gRPC) | `make up-s3` |

> 💡 **Você não precisa de todos os perfis em execução ao mesmo tempo.** Cada célula abaixo é
> independente — execute o `make up-*` do nível que você tem atualmente em execução,
> e seu resultado é anexado a `data/benchmark_results.json`. Volte
> mais tarde com um perfil diferente ativo e adicione ao mesmo arquivo. A célula final
> plota o que foi acumulado até agora.
>
> **Implementação**: veja `scripts/lab_utils.py::run_gold_benchmark`.

### 🧠 O que esperar do benchmark

O `run_gold_benchmark` executa o seguinte pipeline em cada arquitetura:

```python
vendas = spark.read.parquet(...)       # Lê Bronze
empresas = spark.read.parquet(...)     # Lê Bronze (tabela pequena!)
gold = (
    vendas.join(broadcast(empresas))   # Broadcast join (evita shuffle)
    .groupBy("setor", "ano", "mes")    # Agregação
    .agg(sum("valor"))                  # Soma de vendas
)
gold.count()  # Action única — medimos o tempo TOTAL
```

📌 **Dica de análise**: os resultados dependem de:
- **Overhead de inicialização**: YARN precisa iniciar containers; local[*] não
- **Latência de E/S**: volume local > HDFS > S3 (nesta ordem)
- **Paralelismo**: mais núcleos/executores = mais rápido até o ponto de saturação

⚠️ **Cada célula de benchmark é INDEPENDENTE.** Você pode executá-las em
qualquer ordem, em qualquer momento, desde que o perfil Docker necessário
esteja ativo. Os resultados acumulam em `data/benchmark_results.json`.

In [ ]:
import json
import sys
from pathlib import Path

# Adiciona scripts/ ao path e importa as fábricas de sessão e o benchmark
sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, get_local_session, get_yarn_session, run_gold_benchmark

# Arquivo onde os resultados são acumulados (JSON array)
RESULTS_FILE = Path("../data/benchmark_results.json")


def save_result(result):
    """Salva ou atualiza um resultado no arquivo de benchmark.
    Se o label já existir (reexecução), substitui o anterior."""
    # Lê resultados existentes ou inicia array vazio
    results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else []
    # Remove entry com mesmo label se existir (substitui reexecuções)
    results = [r for r in results if r["label"] != result.label]
    # Adiciona novo resultado
    results.append({"label": result.label, "seconds": result.seconds, "row_count": result.row_count})
    # Salva de volta
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f"Saved: {result}")

## Nível A — Local `local[*]`

**Nenhum pré-requisito** — sempre executável.

Esta é a **linha de base** (baseline) do benchmark. O Spark roda no seu próprio
processo, sem Docker, sem rede, sem cluster.

### 🧠 Detalhando o Nível A — Local[*]

**Arquitetura:** Spark em modo `local[*]` — todas as Tasks executam em threads
no mesmo processo JVM. Sem containers, sem rede, sem gerenciador de cluster.

📌 **Características:**
- ✅ Mais rápido (zero overhead de coordenação)
- ✅ Sem dependências (roda em qualquer máquina)
- ✅ Ideal para desenvolvimento e testes
- ❌ Limitado à RAM/CPU de uma máquina
- ❌ Sem tolerância a falhas

📌 **Leitura dos dados:** do disco local via `../data/bronze/vendas/`
(layer_path "local").

**O que esperar:** este deve ser o nível mais rápido, pois não há rede,
serialização ou coordenação de cluster envolvida. A sobrecarga é apenas do
próprio Spark (DAG scheduler, task scheduler, etc.).

> 🎯 **Este é o baseline**: qualquer outro nível será comparado a este.
> A diferença de tempo = custo da distribuição.

In [ ]:
# Cria SparkSession em modo local (sem cluster)
spark = get_local_session("13-benchmark-local")

# Executa o benchmark Gold e salva o resultado
save_result(run_gold_benchmark(spark, "A — local[*]", "local"))

# Libera recursos
spark.stop()

### 📌 Resultado esperado — Nível A

Este benchmark deve ser o mais rápido. Anote o tempo para comparar com os
próximos níveis.

💡 **Como interpretar:**
- Cada segundo extra nos níveis seguintes representa o **custo da distribuição**
- Se o nível A levou 10s e o B levou 30s, o overhead de cluster é ~20s
- Este overhead só se justifica quando os dados não cabem em uma máquina

## Nível B — Spark Standalone + Spark Connect

**Requer `make up-cluster`.**

### 🧠 Detalhando o Nível B — Standalone + Spark Connect

**Arquitetura:**
- **Spark Master** (container `spark-master`) → gerencia o cluster
- **Spark Worker** (container `spark-worker`) → executa Tasks
- **Spark Connect Server** (container `spark-connect`) → recebe comandos gRPC
- **Volume Docker** `/data` → compartilhado entre host e containers

📌 **Fluxo de execução:**
1. Seu notebook envia o DAG via **gRPC** para o Spark Connect Server
2. O Spark Connect Server resolve o plano e submete ao Master
3. O Master agenda as Tasks no Worker
4. O Worker lê os dados do volume compartilhado `/data/bronze/...`
5. Resultado volta pelo mesmo caminho (Worker → Master → Connect → Notebook)

📌 **Características:**
- ✅ Cluster real (mestre+worker)
- ✅ Spark Connect (protocolo gRPC moderno)
- ✅ Volume compartilhado = sem custo de transferência de dados
- ❌ Dados limitados ao tamanho do volume Docker
- ❌ Sem tolerância a falhas (modo Standalone simples)

**O que esperar:** mais lento que local[*] devido ao overhead de:
- Serialização/desserialização gRPC
- Troca de mensagens entre Connect Server → Master → Worker
- Execução em container Docker (virtualização leve)

> 📌 **Compare com o Nível C (YARN)**: aqui não há YARN, então não há overhead
> de negociação de containers — o Spark gerencia seus próprios Workers.

In [ ]:
# Cria SparkSession remota via Spark Connect (gRPC → spark-connect → master → worker)
spark = get_connect_session("13-benchmark-connect")

# Executa o benchmark Gold usando dados do volume Docker (/data/bronze/...)
save_result(run_gold_benchmark(spark, "B — Standalone+Connect", "connect"))

# Libera recursos
spark.stop()

### 📌 Resultado esperado — Nível B

Compare o tempo com o Nível A. A diferença é o **overhead da distribuição**
(geração do DAG + serialização gRPC + agendamento no cluster).

💡 **Observe**: a leitura é do volume Docker (quase tão rápido quanto local),
mas a execução é remota. O overhead aqui é principalmente de **coordenação**,
não de E/S.

## Nível C — YARN + HDFS

**Requer `make up-hadoop`**, e que a escrita Bronze do Lab 08 no HDFS tenha sido executada pelo
menos uma vez (para que `webhdfs://localhost:14000/datalake/bronze/...` exista).

### 🧠 Detalhando o Nível C — YARN + HDFS

**Arquitetura:**
- **YARN ResourceManager** (container `resourcemanager`) → aloca recursos do cluster
- **YARN NodeManagers** (`nodemanager1`, `nodemanager2`) → executam Tasks
- **HDFS** (NameNode + 2 DataNodes) → armazenamento distribuído
- **HttpFS Gateway** (`httpfs`) → ponte HTTP entre host e HDFS

📌 **Fluxo de execução:**
1. Seu Driver (este processo) conecta no ResourceManager
2. O RM aloca um ApplicationMaster e containers executores
3. O Driver lê dados do HDFS via `webhdfs://localhost:14000/datalake/bronze/...`
4. Os dados passam pelo HttpFS Gateway (REST HTTP) e chegam ao Driver
5. O Driver distribui os dados para os executores no YARN
6. Os executores processam em paralelo nos containers

📌 **Características:**
- ✅ Cluster YARN real (ResourceManager + NodeManagers)
- ✅ HDFS com replicação (tolerância a falhas de armazenamento)
- ✅ Suporte a múltiplos frameworks (Spark, MapReduce, Flink...)
- ❌ Custo de E/S: dados passam por HttpFS (REST HTTP) + rede Docker
- ❌ Cold start: YARN precisa iniciar containers (10-20s de overhead)

**O que esperar:** o mais lento dos 4 para este dataset pequeno, porque:
1. YARN precisa iniciar containers (cold start)
2. Dados atravessam HttpFS (REST HTTP → RPC HDFS)
3. Dados trafegam do Driver para os executores

> 📌 **Contra-intuitivo**: YARN + HDFS é o mais "poderoso" em escala, mas o MAIS
> LENTO para datasets pequenos. A tese do laboratório: clusters distribuídos só
> se pagam quando os dados são GRANDES (>10GB).

In [ ]:
# Cria SparkSession conectada ao YARN (client mode)
spark = get_yarn_session("13-benchmark-yarn")

# Executa o benchmark Gold usando dados do HDFS (webhdfs://.../datalake/bronze/...)
save_result(run_gold_benchmark(spark, "C — YARN+HDFS", "hdfs"))

# Libera recursos no YARN
spark.stop()

### 📌 Resultado esperado — Nível C

Este benchmark deve ser o **mais lento** entre os 4 para este dataset pequeno.

📌 **Por quê?**
- YARN negocia containers (ResourceManager + ApplicationMaster) → overhead fixo
- HttpFS adiciona uma camada REST HTTP entre o Driver e o HDFS
- Os dados trafegam: DataNode → HttpFS → Driver → Executor (3 saltos de rede)

💡 **Nota pedagógica importante**: em um cluster real (bare metal) com datasets
grandes (>100GB), o YARN + HDFS frequentemente **supera** o modo local porque:
- A soma da memória/CPU de múltiplos nós é maior que uma máquina
- A localidade de dados (executor perto do bloco) reduz tráfego de rede
- Múltiplos executores processam partitions em paralelo real

## Nível D — Spark Standalone + S3 (RustFS)

**Requer `make up-s3`,** e que a escrita Bronze do Lab 11 em `s3a://bronze/...` tenha
sido executada pelo menos uma vez.

### 🧠 Detalhando o Nível D — Standalone + S3 (RustFS)

**Arquitetura:**
- **Spark Standalone** (Master + Worker) — mesmo cluster do Caso B
- **Spark Connect Server** — mesma conexão gRPC do Caso B
- **RustFS** — servidor de object store compatível com S3 (4 drives + RS(4,2))

📌 **Fluxo de execução:**
1. Seu notebook envia o DAG via gRPC para o Spark Connect Server
2. O Spark Connect Server submete ao Master
3. O Master agenda Tasks no Worker
4. O Worker lê dados do RustFS via **s3a://** (conector S3A + requisições HTTP)
5. Resultado volta pelo mesmo caminho

📌 **Características:**
- ✅ Mesma elasticidade do Spark Standalone
- ✅ Armazenamento desacoplado do processamento (object store)
- ✅ RustFS com Erasure Coding (eficiência de armazenamento)
- ❌ Sem data locality (toda leitura é remota via HTTP)
- ❌ Custo de commit maior (rename é copy+delete no S3)

**O que esperar:** deve ser mais rápido que o YARN (menos overhead de
inicialização) mas mais lento que volume local/Docker devido à latência HTTP
do S3A Connector.

> 📌 **Compare com o Nível C**: ambos têm leitura remota (HDFS via HttpFS vs S3
> via S3A). A diferença principal é o **gerenciador de cluster**:
> - YARN: mais overhead, mas mais recursos de gerenciamento
> - Standalone: mais leve, ideal para ambientes exclusivos Spark

In [ ]:
# Cria SparkSession remota via Spark Connect (mesmo padrão do Caso B)
spark = get_connect_session("13-benchmark-s3")

# Executa o benchmark Gold usando dados do RustFS via s3a://
save_result(run_gold_benchmark(spark, "D — Standalone+S3", "s3"))

# Libera recursos
spark.stop()

### 📌 Resultado esperado — Nível D

Este benchmark deve ficar entre o Nível B (mais rápido, volume compartilhado)
e o Nível C (mais lento, YARN + HDFS via HttpFS).

📌 **Por quê?**
- O RustFS via S3A adiciona latência de requisições HTTP para cada leitura
- Mas não tem o overhead de cold start do YARN (sem containers para iniciar)
- O pipeline: Worker (container) → HTTP → RustFS (container) é uma rede Docker rápida

> 💡 **A surpresa**: dependendo da configuração, o S3 (Nível D) pode ser MAIS
> RÁPIDO que o HDFS (Nível C) neste ambiente dockerizado, porque o RustFS
> não tem a sobrecarga do HttpFS Gateway + handshake RPC do HDFS. Em nuvem,
> a relação se inverte: S3 real tem latência maior que HDFS local.

## O dashboard de comparação

Carrega tudo o que foi acumulado em `data/benchmark_results.json` até agora e
plota — execute esta célula a qualquer momento, mesmo com apenas 1 ou 2 níveis registrados.

### 🎯 Análise guiada do gráfico

Depois de executar a célula abaixo, observe o gráfico de barras e faça estas
perguntas:

1. **Qual nível foi mais rápido?** → Deve ser o A (local[*]), pela ausência de overhead
2. **Qual foi mais lento?** → Provavelmente C (YARN+HDFS), pelo cold start + HttpFS
3. **Qual a diferença entre B e D?** → Ambos usam Spark Connect; a diferença é o
   armazenamento (volume vs S3). Essa diferença = custo do S3A Connector
4. **O ranking faz sentido?** → A < B < D < C (do mais rápido ao mais lento)
   é o esperado para este dataset pequeno

📌 **A grande lição do laboratório:**
> **O valor do Spark não é "mais rápido em qualquer tamanho" — é "a única opção
> quando os dados não cabem mais, ou o trabalho não cabe mais, em uma única máquina."**
>
> Se seus dados cabem em uma máquina, `local[*]` é sempre mais rápido.
> Clusters distribuídos (YARN, Standalone) só se pagam quando a escala exige.

💡 **Experimento extra**: se você tiver tempo, modifique o tamanho do dataset
(em `generate_dataset.py`) para 100M+ linhas e execute os benchmarks novamente.
Você verá o YARN+HDFS se aproximar ou até superar o local[*] conforme o
dataset cresce — porque a memória/CPU combinada do cluster supera a de uma
máquina.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

# Carrega resultados acumulados do arquivo JSON
results = json.loads(Path("../data/benchmark_results.json").read_text())
# Ordena por label para exibição consistente
results.sort(key=lambda r: r["label"])

# Prepara dados para o gráfico
labels = [r["label"] for r in results]
seconds = [r["seconds"] for r in results]

# Cria gráfico de barras
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, seconds, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"][: len(labels)])
ax.set_ylabel("Seconds")
ax.set_title("Same Gold aggregation, 4 architectures")

# Adiciona rótulos com o tempo em segundos acima de cada barra
for bar, s in zip(bars, seconds):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{s:.1f}s", ha="center", va="bottom")

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Interpretando os resultados

Algumas coisas que vale a pena notar (seus números reais variarão por máquina):

- **Local geralmente vence de forma absoluta** neste tamanho de dataset — sem rede,
  sem serialização, sem sobrecarga de coordenação de cluster. Essa sobrecarga só
  se paga quando os dados excedem uma única máquina (regra de ouro de docs/05:
  ~10GB).

- **Os tempos de HDFS e S3 ambos incluem um salto de rede** que os níveis local/volume
  compartilhado não pagam — mas o S3 adicionalmente carece de localidade de dados
  (docs/08), que tende a aparecer mais conforme o tamanho do dataset cresce.

- **Esta é a tese inteira do laboratório**, tornada mensurável: o valor do Spark
  não é "mais rápido em qualquer tamanho" — é "a única opção quando os dados não
  cabem mais, ou o trabalho não cabe mais, em uma única máquina."

> 🎯 **Principais takeaways:**
> 1. **Use local[*] para desenvolvimento** — é rápido, simples e não depende de Docker
> 2. **Use Standalone + volume compartilhado** para testes de integração com cluster
> 3. **Use YARN + HDFS** quando precisar de um cluster multi-framework com tolerância
>    a falhas para datasets grandes
> 4. **Use Standalone + S3** quando precisar de elasticidade e desacoplamento
>    armazenamento-processamento, mesmo que pague o preço em latência
>
> 💡 **O melhor engenheiro de dados não é o que faz o Spark rodar mais rápido —
> é o que sabe qual ferramenta usar para cada escala de problema.**